# 💎 Phainora Gemma على Kaggle — Gemma 4 E4B بيتعلم يعمل مواقع بأسلوبك

Kaggle بدل Colab عشان فيه ~30 جيجا رامة (Gemma 4 E4B وقّع رامة Colab المجانية).

## الخطوات
1. على اليمين: **Settings ← Accelerator ← GPU T4 x2**، و **Internet ← On**.
2. **Run All**. حوالي ساعة أو اتنين.
3. في الآخر من **Output** (على اليمين) ← `weblume_gemma` نزّل `phainora-gemma.gguf` و `Modelfile`.


## ⚙️ الإعدادات

In [ ]:
#@title الإعدادات
# Gemma 4 E4B ← نفس اللي على جهازك في Ollama (gemma4:latest، ≈ 6.6 جيجا Q4)
BASE_MODEL = "unsloth/gemma-4-E4B-it-unsloth-bnb-4bit"  #@param ["unsloth/gemma-4-E4B-it-unsloth-bnb-4bit"]  # already 4-bit: the full-size one ran out of Colab RAM
EPOCHS = 1     #@param {type:"integer"}
# لو فيه تدريب قديم في Drive يكمّل عليه (أسرع بكتير) ويتدرّب على نص البيانات بس
CONTINUE_TRAINING = False  #@param {type:"boolean"}
MAX_LEN = 16384

SYSTEM_PROMPT = (
    "اسمك Weblume AI، مساعد شركة Weblume لعمل المواقع. انت مبرمج Full-Stack محترف ومصمم واجهات مبدع، "
    "بتتكلم مصري بسيط وودود، والمستخدم ممكن يكون مبتدئ.\n"
    "- لو ماحددش لغة برمجة: اكتب موقع HTML واحد كامل فيه CSS وJS، شغال على الموبايل، من غير مكتبات لازمة.\n"
    "- التصميم لازم يبقى مبتكر واحترافي: هوية بصرية خاصة بالمشروع، خطوط Google Fonts مميزة، "
    "مسافات مريحة، أنيميشن ناعم، وتفاصيل صغيرة تفرق. ممنوع التصميم العادي المكرر.\n"
    "- النصوص حقيقية ومقنعة بلغة الموقع، ومن غير Lorem ipsum. العربي RTL.\n"
    "- في التعديل: نفّذ المطلوب بالظبط من غير ما تبوّظ باقي الموقع، ورجّع الملف كامل بعد التعديل.\n"
    "- اكتب الكود جوه ```html وبعده نقط قصيرة باللي عملته.\n"
    "- الصور: <img src=\"https://placehold.co/WxH\" data-photo=\"english search words\" alt=\"وصف عربي\" width=\"W\" height=\"H\">، وماتألفش لينكات صور.\n"
    "- ماتكتبش أرقام تليفونات من عندك؛ لو مش متاحة حط مكانها واضح للتعديل."
)
print("✅", BASE_MODEL, "| Epochs:", EPOCHS)

## 1️⃣ كارت الشاشة + الأدوات


In [ ]:
import subprocess, os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Unsloth trains on one card; Kaggle gives two
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
if gpu.returncode != 0:
    raise SystemExit("❌ مفيش كارت شاشة! Settings ← Accelerator ← GPU T4 x2، وبعدين Run All.")
print("✅ كارت الشاشة:", gpu.stdout.strip())
WORK = "/kaggle/working/weblume_gemma"  # Kaggle's Output folder
os.makedirs(WORK, exist_ok=True)
DATA_FILE = f"{WORK}/train_data.jsonl"
print("📁 الشغل بيتحفظ في:", WORK)


In [ ]:
%%capture
!pip install -q unsloth

In [ ]:
from unsloth import FastModel  # own cell, right after the install (same cell or later crashed the Colab VM)

## 2️⃣ بيانات التدريب
اتكتبت بأسلوب Weblume: ١٤٠ موقع لـ٢٤ نوع نشاط (كافيهات ومطاعم وعيادات وجيمات وبراندات وشركات وبورتفوليو وتطبيقات)، و١٢ ستايل ألوان، و٤ أشكال هيرو، و٧٠ تعديل، و٤٠ سكشن لوحده، وأسئلة برمجة، وردود عملاء، **وموقع Weblume بتاعك**.

In [ ]:
#@title جهّز بيانات التدريب (مكتوبة بأسلوب Weblume)
import base64, io, zipfile, sys, json, random, importlib
import urllib.request
GEN_ZIP = urllib.request.urlopen("https://raw.githubusercontent.com/weblume-ai/weblume-ai.github.io/main/training/data/gen_zip.b64").read().decode()  # kept outside: Kaggle notebooks must be under 1 MB
zipfile.ZipFile(io.BytesIO(base64.b64decode("".join(GEN_ZIP.split())))).extractall("/content/weblume_gen")
sys.path.insert(0, "/content/weblume_gen")
import dataset; importlib.reload(dataset)
DATA = dataset.make_dataset() + dataset.mine_examples("/content/weblume_gen/mine")
from collections import Counter
print("✅ بيانات التدريب:", len(DATA), dict(Counter(d["kind"] for d in DATA)))

In [ ]:
#@title مواقع Phainora المعتمدة (96 مواقع كاملة بأنيميشن وعربي/إنجليزي)
# كل موقع بيتكرر مرتين علشان الموديل ياخد المستوى ده. الصور فيها صيغة ثابتة يتبدل مكانها صور حقيقية بعدين.
import base64, io, zipfile, json, os
import urllib.request
PHAINORA_ZIP = urllib.request.urlopen("https://raw.githubusercontent.com/weblume-ai/weblume-ai.github.io/main/training/data/phainora_zip.b64").read().decode()  # kept outside: Kaggle notebooks must be under 1 MB
os.makedirs("/content/phainora", exist_ok=True)
zipfile.ZipFile(io.BytesIO(base64.b64decode("".join(PHAINORA_ZIP.split())))).extractall("/content/phainora")
for ex in json.load(open("/content/phainora/examples.json", encoding="utf-8")):
    html = open("/content/phainora/" + ex["file"], encoding="utf-8").read().strip()
    ans = f"تمام! ده الموقع كامل في ملف واحد 👇\n\n```html\n{html}\n```\n\n- الصور مكتوبة بالصيغة الثابتة (data-photo)، وسكريبت الصور بيحط مكانها صور حقيقية.\n- زرار EN بيقلب الموقع إنجليزي.\nلو حابب أغيّر أي حاجة قولّي."
    DATA += [{"kind": "mine", "messages": [{"role": "user", "content": ex["request"]}, {"role": "assistant", "content": ans}]}] * 2
print("✅ اتضاف", len(json.load(open("/content/phainora/examples.json", encoding="utf-8"))) * 2, "مثال من مواقع Phainora — الإجمالي", len(DATA))


### مواقعك التانية (اختياري)
لو عندك مواقع تانية عملتها (ملف HTML واحد)، فعّل الاختيار وارفعها، والموديل هيتعلم منها أسلوبك أكتر.

In [ ]:
#@title ارفع مواقع زيادة
UPLOAD_MY_SITES = False  # Colab only
if UPLOAD_MY_SITES:
    from google.colab import files
    import os, re
    extra = []
    for name, raw in files.upload().items():
        html = raw.decode("utf-8", "ignore")
        if "<html" not in html.lower(): print("⚠️ اتجاهل:", name); continue
        html = re.sub(r"data:([a-z/+]+);base64,[A-Za-z0-9+/=]{100,}", "assets/image.png", html)
        open(f"/content/weblume_gen/mine/{name}", "w", encoding="utf-8").write(html)
        title = (re.findall(r"<title>(.*?)</title>", html, re.S) or [name])[0].strip()
        extra.append((name, f"اعملي موقع لـ{title} بأسلوب Weblume"))
    DATA += dataset.mine_examples("/content/weblume_gen/mine", extra)[len(dataset.MINE) * 3:]
    print("✅ بقى عندك", len(DATA), "مثال")

In [ ]:
#@title شوف عينة من المواقع اللي هيتعلم منها
import re, html as _h
from IPython.display import HTML, display
def html_of(text):
    m = re.search(r"```html\s*(.*?)```", text, re.S | re.I)
    return m.group(1).strip() if m else ""
s = random.choice([d for d in DATA if d["kind"] == "site"])
print("🧑", s["messages"][0]["content"])
display(HTML(f'<iframe srcdoc="{_h.escape(html_of(s["messages"][1]["content"]))}" style="width:100%;height:560px;border:1px solid #333;border-radius:8px"></iframe>'))

## 4️⃣ التدريب (LoRA على كارت T4 المجاني)

In [ ]:
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template
import torch, glob, random
from datasets import Dataset

CK_DIR = f"{WORK}/checkpoints_g{len(DATA)}"  # a new folder per data version; a crash resumes from here
model, tokenizer = FastModel.from_pretrained(model_name=BASE_MODEL, max_seq_length=MAX_LEN, load_in_4bit=True, full_finetuning=False)
model = FastModel.get_peft_model(
    model, finetune_vision_layers=False, finetune_language_layers=True, finetune_attention_modules=True, finetune_mlp_modules=True,
    r=16, lora_alpha=16, lora_dropout=0, bias="none", random_state=3407, use_gradient_checkpointing="unsloth")
tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")  # the non-thinking template for the small Gemma 4s (Unsloth guide)
tok = getattr(tokenizer, "tokenizer", tokenizer)  # the text tokenizer inside the processor

# each Phainora site once (they're listed twice for the Qwen run), all the chat replies, a quarter of the generated examples: ~1 h on a T4
rnd, seen, TRAIN = random.Random(7), set(), []
for d in DATA:
    key = d["messages"][0]["content"]
    if d["kind"] == "mine":
        if key in seen:
            continue
        seen.add(key)
        TRAIN.append(d)
    elif d["kind"] == "chat" or rnd.random() < 0.25:
        TRAIN.append(d)


def fmt(ex):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}] + ex["messages"]
    msgs = [{"role": m["role"], "content": [{"type": "text", "text": m["content"]}]} for m in msgs]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False).removeprefix("<bos>")


texts = [fmt(e) for e in TRAIN]
keep = [t for t in texts if len(tok(t, add_special_tokens=False)["input_ids"]) <= MAX_LEN]
print(f"✅ {len(keep)} مثال جاهز للتدريب (اتشال {len(texts) - len(keep)} عشان أطول من {MAX_LEN} توكن)")
print(texts[0][:300])
ds = Dataset.from_list([{"text": t} for t in keep])


In [ ]:
from trl import SFTTrainer, SFTConfig
cfg = dict(dataset_text_field="text", max_seq_length=MAX_LEN, max_length=MAX_LEN, per_device_train_batch_size=1,
           gradient_accumulation_steps=8, num_train_epochs=EPOCHS, learning_rate=2e-4, warmup_steps=3,
           logging_steps=5, optim="adamw_8bit", weight_decay=0.01, lr_scheduler_type="cosine",
           seed=3407, output_dir=CK_DIR, report_to="none", save_strategy="steps", save_steps=4, save_total_limit=1,
           fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported())
try:
    args = SFTConfig(**cfg)
except TypeError:  # نسخ trl بتقبل اسم واحد بس من الاتنين
    try:
        cfg.pop("max_seq_length"); args = SFTConfig(**cfg)
    except TypeError:
        cfg["max_seq_length"] = cfg.pop("max_length"); args = SFTConfig(**cfg)
try:
    trainer = SFTTrainer(model=model, processing_class=tokenizer, train_dataset=ds, args=args)
except TypeError:
    trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=ds, args=args)
try:  # يتعلم من الردود بس، مش من الأسئلة
    from unsloth.chat_templates import train_on_responses_only
    trainer = train_on_responses_only(trainer, instruction_part="<|turn>user\n", response_part="<|turn>model\n")
except Exception as e:
    print("ℹ️ هندرّب على النص كله:", e)

import glob as _g
resume = bool(_g.glob(f"{CK_DIR}/checkpoint-*"))
if resume: print("↩️ بكمّل التدريب من آخر نقطة اتحفظت")
stats = trainer.train(resume_from_checkpoint=resume)
print(f"✅ خلص التدريب في {stats.metrics['train_runtime']/60:.1f} دقيقة — loss: {stats.metrics['train_loss']:.3f}")

## 5️⃣ جرّب الموديل بنفسك

In [ ]:
FastModel.for_inference(model)
def ask(q, max_new_tokens=6000):
    msgs = [{"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]}, {"role": "user", "content": [{"type": "text", "text": q}]}]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = tok(text, return_tensors="pt", add_special_tokens=False)["input_ids"].to("cuda")
    out = model.generate(input_ids=ids, max_new_tokens=max_new_tokens, temperature=1.0, top_p=0.95, top_k=64, do_sample=True)
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True)

def show(answer):
    h = html_of(answer)
    print(re.sub(r"```html.*?```", "[… الكود …]", answer, flags=re.S)[:1500])
    if h:
        display(HTML(f'<iframe srcdoc="{_h.escape(h)}" style="width:100%;height:560px;border:1px solid #333;border-radius:8px"></iframe>'))
        open("test_site.html", "w", encoding="utf-8").write(h)

print("⏳ بيكتب موقع كامل… (كام دقيقة على كارت Colab)")
show(ask("اعملي موقع لمحل كنافة وحلويات شرقية اسمه Sweet Nile، ستايل داكن فخم وفيه منيو وطلب واتساب"))


In [ ]:
#@title اطلب أي موقع
طلبك = "اعملي لاندنج بيدج لجيم اسمه Iron House بالإنجليزي، ستايل نيون"  #@param {type:"string"}
show(ask(طلبك))

## 6️⃣ تصدير الموديل لـ Ollama
Unsloth بيدمج التدريب في الموديل الكامل (16-bit) وبعدين يضغطه **مرة واحدة** لـ Q4_K_M، فالتدريب مايضيعش.


In [ ]:
import glob, os, shutil
model.save_pretrained(f"{WORK}/lora"); tokenizer.save_pretrained(f"{WORK}/lora")  # small backup (~100 MB)
OUT = "/content/gguf"
model.save_pretrained_gguf(OUT, tokenizer, quantization_method="q4_k_m")
g = [f for f in glob.glob(f"{OUT}*/*.gguf") + glob.glob(f"{OUT}*.gguf") if "mmproj" not in f.lower()]
if not g:
    raise SystemExit("❌ التصدير فشل — ابعت صورة للرسالة اللي فوق")
shutil.copy(max(g, key=os.path.getsize), f"{WORK}/phainora-gemma.gguf")
modelfile = ('FROM ./phainora-gemma.gguf\n\nSYSTEM """' + SYSTEM_PROMPT.replace('"""', "'") + '"""\n\n'
             "PARAMETER temperature 1.0\nPARAMETER top_p 0.95\nPARAMETER top_k 64\nPARAMETER num_ctx 16384\nPARAMETER num_predict 12000\n")
open(f"{WORK}/Modelfile", "w", encoding="utf-8").write(modelfile)
print("✅ جاهز:", round(os.path.getsize(f"{WORK}/phainora-gemma.gguf") / 1e9, 2), "جيجا ← Output/weblume_gemma (phainora-gemma.gguf + Modelfile)")


In [ ]:
print("📁 نزّل phainora-gemma.gguf و Modelfile من Output ← weblume_gemma (على اليمين)")


## ✅ خلصنا!
من **Output ← weblume_gemma** نزّل `phainora-gemma.gguf` و `Modelfile` وحطهم في فولدر واحد، وبعدين:
`ollama create phainora-gemma -f Modelfile`
